In [2]:
import pandas as pd
import sqlite3

df = pd.read_csv("cleaned_sales_data.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Product Name,Sales,Quantity,Discount,Profit,Profit Margin,Order Year,Order Month,Order Month Name,Discount Category
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,0.1600,2016,11,Nov,No Discount
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,0.3000,2016,11,Nov,No Discount
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,0.4700,2016,6,Jun,No Discount
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,-0.4000,2015,10,Oct,High Discount
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,0.1125,2015,10,Oct,High Discount


In [3]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

df.isnull().sum()

,0
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


customer-level data

In [4]:
customer_df = df.groupby(['Customer ID', 'Customer Name', 'Segment', 'Region']).agg(
    total_sales=('Sales', 'sum'),
    total_profit=('Profit', 'sum'),
    order_count=('Order ID', 'nunique'),
    last_order_date=('Order Date', 'max')
).reset_index()

customer_df.head()

,Customer ID,Customer Name,Segment,Region,total_sales,total_profit,order_count,last_order_date
0,AA-10315,Alex Avila,Consumer,Central,4780.552,-650.5971,2,2017-06-29
1,AA-10315,Alex Avila,Consumer,East,29.500,13.2826,1,2014-09-15
2,AA-10315,Alex Avila,Consumer,West,753.508,274.4320,2,2015-10-04
3,AA-10375,Allen Armold,Consumer,Central,178.370,62.0658,1,2015-02-03
4,AA-10375,Allen Armold,Consumer,East,206.732,68.9195,2,2017-12-11


recency

In [5]:
reference_date = df['Order Date'].max()

customer_df['recency_days'] = (reference_date - customer_df['last_order_date']).dt.days
customer_df['avg_order_value'] = customer_df['total_sales'] / customer_df['order_count']

customer segments

In [6]:
def customer_value_segment(row):
    if row['total_sales'] >= customer_df['total_sales'].quantile(0.75):
        return 'High Value'
    elif row['total_sales'] >= customer_df['total_sales'].quantile(0.40):
        return 'Medium Value'
    else:
        return 'Low Value'

customer_df['customer_value_segment'] = customer_df.apply(customer_value_segment, axis=1)

recency segment

In [7]:
def recency_segment(days):
    if days <= 90:
        return 'Active'
    elif days <= 180:
        return 'At Risk'
    else:
        return 'Inactive'

customer_df['recency_segment'] = customer_df['recency_days'].apply(recency_segment)

In [15]:
customer_df['customer_value_segment'].value_counts()

,count
customer_value_segment,
Low Value,1000
Medium Value,875
High Value,626


In [16]:
customer_df['recency_segment'].value_counts()

,count
recency_segment,
Inactive,1586
Active,576
At Risk,339


In [8]:
customer_df.to_csv("customer_segmentation_data.csv", index=False)